# Funkce hustoty pravděpodobnosti

- Každé měření libovolné složitosti si lze představit jako losování jednoho nebo více náhodných čísel z nějaké pravděpodobnostní distribuce (PDF).

- I když zpracujete obrovský soubor dat, ale měříte pouze jednu veličinu, stále to je jako losování jednoho náhodného čísla z nějaké PDF.

- Představte si, že měříte hmotnost nějakého předmětu. Uděláte 1000 vážení a dostanete 1000 různých hodnot. Výpočet průměru je ekvivalentní vylosování jednoho náhodného čísla z nějaké PDF. Průměr je náhodná veličina, protože závisí na 1000 náhodných číslech, která jste dostali z tisícinásobného vážení.

- Modelujme jedno vážení normálním rozdělením s $\mu = 100$ g a $\sigma = 5$ g; tuto PDF si nazveme "PDF váhy".

In [ ]:
# Jednoduchý model vážení pomocí váhy: normální rozdělení s průměrem 100 g a směrodatnou odchylkou 5 g.
from scipy.stats import norm
import numpy as np
import matplotlib.pyplot as plt
mu = 100
sigma = 5

x = np.linspace(80, 120, 2000)
balance_pdf = norm.pdf(x, loc = mu, scale = sigma)
plt.plot(x, balance_pdf)
plt.xlabel(r'$m$ [g]')
plt.ylabel('Hustota pravděpodobnosti')
plt.title('PDF váhy')
plt.show()

- Zvážení předmětu 1000krát odpovídá losování 1000 náhodných čísel z PDF váhy. Každou hodnotu zobrazme jako bod ve "scatter plotu".

In [ ]:
# Vylosujeme 1000 náhodných čísel z PDF váhy.
N = 1000
samples = norm.rvs(loc = mu, scale = sigma, size = N)

# Zobrazíme scatter plot 1000 hodnot.
plt.scatter(range(N), samples, alpha = 0.5)
# Vyznačíme interval +- 1 sigma horizontálními čarami.
plt.axhline(y = mu + sigma, color = 'r', linestyle = '--', label = r'$1 \sigma$')
plt.axhline(y = mu - sigma, color = 'r', linestyle = '--')
# Zapíšeme +- 1 sigma do grafu.
plt.text(0, mu + sigma + 0.5, r'$\mu + 1\sigma$', color = 'r')
plt.text(0, mu - sigma - 1.5, r'$\mu - 1\sigma$', color = 'r')
# Zapíšeme +- 2 sigma do grafu.
plt.axhline(y = mu + 2 * sigma, color = 'g', linestyle = '--', label = r'$2 \sigma$')
plt.axhline(y = mu - 2 * sigma, color = 'g', linestyle = '--')
plt.text(0, mu + 2 * sigma + 0.5, r'$\mu + 2\sigma$', color = 'g')
plt.text(0, mu - 2 * sigma - 1.5, r'$\mu - 2\sigma$', color = 'g')
plt.xlabel('Číslo vzorku')
plt.ylabel(r'$m$ [g]')
plt.title('Scatter plot 1000 vzorků z PDF váhy')
plt.show()

- Hmotnost odhadneme jako průměr $N = 1000$ hodnot $\{m_i\}$.

$$
\hat{m} = \frac{1}{N} \sum_{i=1}^N m_i
$$

- Neurčitost průměru odhadneme obvyklým způsobem:

$$
\hat{\sigma}_{\hat{m}} = \frac{\hat{\sigma}}{\sqrt{N}} = \frac{1}{\sqrt{N}} \sqrt{\frac{1}{N-1} \sum_{i=1}^N (m_i - \hat{m})^2}
$$
    
- Poznámka: $\hat{\sigma}$ je odhad parametru PDF váhy, zatímco $\hat{\sigma}_{\hat{m}}$ je neurčitost odhadu průměru, $\hat{m}$.

In [ ]:
# Odhadneme průměr a jeho neurčitost.
mhat = np.mean(samples)
sigmahat_m = np.std(samples, ddof = 1) / np.sqrt(len(samples))
print(f'Odhadovaný průměr: mhat = {mhat:.2f} g')
print(f'Odhadovaná neurčitost průměru: sigmahat_m {sigmahat_m:.2f} g')

- PDF pro $\hat{m}$ je normální rozdělení s $\mu' = 100$ g a $\sigma' =  \sigma / \sqrt{N}$.

- Nyní zobrazíme PDF pro $\hat{m}$ společně s PDF váhy v jednom grafu, abychom viděli, že je její šířka o hodně menší než šířka PDF váhy.

In [ ]:
# Zobrazíme PDF pro mhat a PDF váhy.
mhat_pdf = norm.pdf(x, loc = mu, scale = sigma / np.sqrt(len(samples)))
plt.plot(x, balance_pdf, label = 'PDF váhy')
plt.plot(x, mhat_pdf, label = r'PDF pro $\hat{m}$')
plt.xlabel(r'$m$ [g]')
plt.ylabel('Hustota pravděpodobnosti')
plt.title(r'PDF váhy a PDF pro $\hat{m}$')
plt.legend()
plt.show()

Shrnutí: celé měření je vlastně jako vylosování jednoho náhodného čísla ($\hat{m}$) z nějaké PDF (PDF pro $\hat{m}$). Díky velkému počtu (1000) vážení, která byla provedena, je PDF pro $\hat{m}$ výrazně užší než PDF váhy. Odhad neurčitosti $\hat{m}$ je proto výrazně menší než parametr $\sigma$ PDF váhy.

# PDF v SciPy

- SciPy obsahuje obrovský počet PDF implementovaných v modulu `scipy.stats`. Úplný seznam najdete [zde](https://docs.scipy.org/doc/scipy/reference/stats.html).

- Každá PDF je implementována jako třída s metodami pro vyhodnocení PDF, CDF a dalších vlastností. Například normální rozdělení je implementováno jako `scipy.stats.norm`. Podívejme se, jak ho používat:

In [ ]:
from scipy.stats import norm
# Vyhodnotíme PDF normálního rozdělení pro tři hodnoty proměnné x.
x_values = [2, 4, 5]
pdf_values = norm.pdf(x_values, loc = 5, scale = 3)
print(f'Hodnoty PDF pro x = {x_values}: {pdf_values}')

- Parametry `loc` a `scale` jsou důležité! Většina metod tříd PDF má tyto parametry. Jejich přesný význam závisí na konkrétní PDF. Jsou definovány tak, že

$$
\mathrm{pdf}(x, \mathrm{loc}, \mathrm{scale})\ \equiv\ \frac{1}{\mathrm{scale}} \mathrm{pdf}\left(\frac{x - \mathrm{loc}}{\mathrm{scale}}\right)
$$

- Pro normální rozdělení je `loc` střední hodnota, $\mu$, a `scale` je šířka, $\sigma$.

- Kumulativní distribuční funkce (CDF) je integrál PDF od $-\infty$ do $x$:
$$
\mathrm{cdf}(x, \mathrm{loc}, \mathrm{scale})\ \equiv\ \mathrm{cdf}\left(\frac{x - \mathrm{loc}}{\mathrm{scale}}\right)
$$

In [ ]:
# Vyhodnotíme CDF normálního rozdělení pro tři hodnoty proměnné x.
cdf_values = norm.cdf(x_values, loc = 5, scale = 3)
print(f'Hodnoty CDF pro x = {x_values}: {cdf_values}')

- Pro získání percentilů použijeme funkci `ppf`, která je inverzí CDF:

In [ ]:
# Vyhodnotíme 16., 50. a 84. percentil normálního rozdělení.
integrals = [0.16, 0.50, 0.84]
percentiles = norm.ppf(integrals, loc = 5, scale = 3)
print(f'Percentily pro {integrals}: {percentiles}')

- Důležitou funkcionalitou tříd PDF je možnost vylosovat náhodná čísla z PDF. Provádí se pomocí metody `rvs`, která označuje "náhodné výběry" (random variates). Například 10 náhodných čísel vylosujeme takto:

In [ ]:
# Vylosujeme 10 náhodných čísel a vypíšeme je.
random_numbers = norm.rvs(loc = 5, scale = 3, size = 10)
print(random_numbers)

# PDF v NumPy

- NumPy neposkytuje analytické PDF, takže nelze například vyhodnotit PDF nebo CDF v nějakém bodě $x$.

- Lze však vylosovat náhodná čísla z PDF pomocí modulu `numpy.random`. Například:

In [ ]:
# Vylosujeme 10 náhodných čísel z normálního rozdělení.
import numpy as np
random_numbers = np.random.normal(loc = 5, scale = 3, size = 10)
print(random_numbers)

- Parametry `loc` a `scale` jsou stejné jako v SciPy.

- NumPy poskytuje metody pro datovou analýzu vylosovaných náhodných čísel. Např. se jedná o `numpy.mean` a `numpy.std`, které lze použít pro odhad střední hodnoty a směrodatné odchylky PDF, ze které jste čísla losovali.

In [ ]:
# Odhadneme střední hodnotu a směrodatnou odchylku vylosovaných náhodných čísel.
estimated_mean = np.mean(random_numbers)
estimated_std = np.std(random_numbers, ddof = 1)
print(f'Odhadovaná střední hodnota: {estimated_mean:.2f}')
print(f'Odhadovaná směrodatná odchylka: {estimated_std:.2f}')

# Vícerozměrné PDF

- SciPy také poskytuje vícerozměrné PDF, jako je [scipy.stats.multivariate_normal](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.multivariate_normal.html), což je vícerozměrné normální rozdělení. Parametry vícerozměrného normálního rozdělení jsou vektor střední hodnoty a kovarianční matice.

- Vzorec pro vícerozměrné normální rozdělení obsahuje kovarianční matici:
$$
\mathrm{pdf}(\mathbf{x}, \boldsymbol{\mu}, \boldsymbol{V})\ \equiv\ \frac{1}{(2\pi)^{k/2} |\boldsymbol{V}|^{1/2}} \exp\left(-\frac{1}{2} (\mathbf{x} - \boldsymbol{\mu})^T \boldsymbol{V}^{-1} (\mathbf{x} - \boldsymbol{\mu})\right)
$$

- Normální rozdělení je jediné rozdělení, které je plně popsáno svou střední hodnotou a kovariancí. Ostatní PDF potřebují pro úplný popis více momentů (obecně nekonečně mnoho).

In [ ]:
# Zobrazíme 2D normální rozdělení s vektorem střední hodnoty [0, 0] a kovarianční maticí [[3, 2], [2, 2]].
from scipy.stats import multivariate_normal
import numpy as np
import matplotlib.pyplot as plt
mean = [0, 0]
cov = [[3, 2], [2, 2]]
x = np.linspace(-3, 3, 100)
y = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x, y)
pos = np.dstack((X, Y))
Z = multivariate_normal.pdf(pos, mean, cov)
plt.contourf(X, Y, Z, levels = 50, cmap = 'viridis')
plt.colorbar(label = 'Hustota pravděpodobnosti')
plt.xlabel('x')
plt.ylabel('y')
plt.title('PDF vícerozměrného normálního rozdělení')
plt.show()

# Odhad kovarianční matice

- Kovarianční matice je zobecnění variance do více dimenzí. Popisuje varianci dat ve více dimenzích a korelace mezi nimi.

- Při měření více veličin je třeba odhadnout kovarianční matici, protože obsahuje informace o neurčitostech (a to nejen její diagonální prvky, nýbrž všechny prvky).

- Pokud vylosujeme několik dvojic hodnot ze 2D rozdělení, pak se kovarianční matice odhadne takto:
    $$
    \hat{V}_{ij} = \frac{1}{N-1} \sum_{k=1}^N (x_{i,k} - \hat{\mu}_i) (x_{j,k} - \hat{\mu}_j)
    $$
    kde $x_{i,k}$ je $i$-tá složka $k$-té vylosované dvojice a $\hat{\mu}_i$ je odhadovaná střední hodnota $i$-té složky.

- Vysvětlení: výše uvedený vzorec aproximuje teoretický výpočet, který by sice byl ideální, ale nelze jej provést, protože neznáme správnou PDF (přinejmenším neznáme hodnoty jejích parametrů). Ve výpočtu mají body velkou váhu, pokud se nacházejí v oblasti s vysokou hustotou pravděpodobnosti:
    $$
    V_{ij} = \int \Pi_k \mathrm{d}x_k (x_i - \mu_i) (x_j - \mu_j) \mathrm{pdf}(x_k)
    $$
    Pokud vylosujeme dvojice hodnot z PDF, dostaneme více bodů v oblasti s vysokou hustotou pravděpodobnosti, takže můžeme aproximovat integrál součtem přes vylosované dvojice, což je výše uvedený vzorec.

- V mnoha případech však nemáme pro tento odhad kovarianční matice data. Například při fitování modelu na data máme jako výsledek pouze jednu jedinou dvojici (nebo obecně jeden N-tici) hodnot. Naštěstí existují jiné metody pro odhad kovarianční matice, jako jsou Fisherova nebo Hessova matice (druhé derivace likelihoodu nebo loss funkce). Tato technika je diskutována [jinde](https://youtu.be/xwcVRAUpRkU?si=jCMjxxMH9hOHph0m), ale je ve fyzice hojně používána. Mějte na paměti, že nástroje jako `scipy.optimize.curve_fit` tuto, nebo nějakou podobnou metodu využívají.

- V některých případech nefunguje žádná analytická metoda... Naštěstí máme Monte Carlo metody, které jsou založeny na generování náhodných čísel! Viz konec tohoto notebooku.

# Použití kovarianční matice

- Kdykoli vyhodnocujete výraz závislý na několika náhodných proměnných, $y = y(x_1, x_2, \ldots, x_n)$, musíte použít kovarianční matici pro výpočet neurčitosti daného výrazu. To se provádí pomocí notoricky známého vzorce pro propagaci neurčitostí:
  $$
  \sigma_{\hat{y}} = \sqrt{\sum\limits_{i,j=1}^n \frac{\partial y}{\partial x_i}\Big|_{\vec x=\hat{\vec{x}}} \frac{\partial y}{\partial x_j}\Big|_{\vec x=\hat{\vec{x}}} V_{ij}}
  $$
  V případě dvou proměnných tento vzorec vypadá takto:
    $$
    \sigma_{\hat{y}} = \sqrt{\left(\frac{\partial y}{\partial x_1}\Big|_{\vec x=\hat{\vec{x}}}\right)^2 V_{11} + \left(\frac{\partial y}{\partial x_2}\Big|_{\vec x=\hat{\vec{x}}}\right)^2 V_{22} + 2 \frac{\partial y}{\partial x_1}\Big|_{\vec x=\hat{\vec{x}}} \frac{\partial y}{\partial x_2}\Big|_{\vec x=\hat{\vec{x}}} V_{12}}
    $$

- V praxi vždy používejte nástroje, které tento vzorec implementují, např. balík `uncertainties` v Pythonu. Ten se postará o derivování a všechno ostatní za vás. Jako příklad předpokládejme, že jsme změřili délku a šířku obdélníku a chceme vypočítat jeho plochu a odhadnou její neurčitost (nebavme se teď o tom, kde se vzala korelace mezi délkou a šířkou):

In [ ]:
import uncertainties

nominal_values    = [1, 2]
covariance_matrix = [[0.01, -8.e-3 ],
                     [-8.e-3 , 0.03]]
(a, b) = uncertainties.correlated_values(nominal_values, covariance_matrix)
print(a * b)

# Vypočítejme to samé, přičemž korelaci mezi dvěma proměnnými ignorujeme.
a_uncorr = uncertainties.ufloat(a.nominal_value, a.std_dev)
b_uncorr = uncertainties.ufloat(b.nominal_value, b.std_dev)
print(a_uncorr * b_uncorr)

# Pseudoexperimenty (Monte Carlo metoda)

- V některých případech nefunguje žádná analytická metoda pro odhad požadovaných veličin (např. kovarianční matice). V takových případech můžeme použít metodu Monte Carlo, která je založena na generování náhodných čísel.

- Výhody:

  - Má velmi málo předpokladů.

  - Funguje vždy, když jde losovat náhodná čísla z pravděpodobnostního rozdělení měřených veličin a vyhodnotit hledaný výraz.

- Nevýhody:

  - Je výpočetně náročná, protože je třeba losovat velké množství náhodných čísel a pro každé losování vyhodnotit zadaný výraz.

- Nejlépe se metoda pseudoexperimentů vysvětlí na příkladu. Řekněme, že jsme změřili délku strany čtverce $\hat{a}$ a její neurčitost $\hat{\sigma}_{\hat{a}}$ a chceme vypočítat plochu čtverce a její neurčitost.

  - Vygenerujeme sadu pseudoměření délky $\{\hat{a}_i\}_{i=1}^N$ losováním náhodných čísel z PDF pro $\hat{a}$.

  - PDF pro $\hat{a}$ je normální rozdělení se střední hodnotou $a$ a směrodatnou odchylkou $\sigma_a$. Jelikož neznáme skutečné hodnoty $a$ a $\sigma_a$, použijeme odhady $\hat{a}$ a $\hat{\sigma}_{\hat{a}}$.

  - Pro každé pseudoměření $\hat{a}_i$ vypočítáme plochu čtverce $\hat{A}_i = \hat{a}_i^2$.

  - Odhadneme varianci PDF plochy jako varianci souboru $\{\hat{A}_i\}_{i=1}^N$:

  $$
  \hat{\sigma}_A^2 = \frac{1}{N} \sum_{i=1}^N (\hat{A}_i - \hat{A})^2
  $$
  kde $\hat{A} = \hat{a}^2$ je náš odhad plochy na základě naměřené délky $\hat{a}$.

In [ ]:
# Kód pro výše uvedený příklad.
import numpy as np

# Nastavíme číselné hodnoty pro naměřenou délku a její neurčitost.
ahat = 5
sigmahat_a = 2
N = 1000

# Vygenerujeme N pseudoexperimentů.
a_pseudo = np.random.normal(ahat, sigmahat_a, N)

# Vypočítáme plochy pro každý pseudoexperiment.
A_pseudo = a_pseudo ** 2

# Odhadneme střední hodnotu a směrodatnou odchylku plochy.
Ahat = np.mean(A_pseudo)
sigmahat_A = np.std(A_pseudo, ddof = 0)

print(f'Odhadovaná neurčitost plochy: {sigmahat_A:.2f} m^2')

# Porovnáme s vzorcem pro propagaci neurčitostí.
import uncertainties
a = uncertainties.ufloat(ahat, sigmahat_a)
A = a ** 2
print(f'Odhadovaná neurčitost plochy pomocí vzorce pro propagaci neurčitostí: {A.std_dev:.2f} m^2')

Další příklad: Odhad kovarianční matice v úloze "Absorpce beta záření" pomocí pseudoexperimentů.

- Začněme řešením dané domácí úlohy:

In [ ]:
import matplotlib.pyplot as plt
import uncertainties
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

data_str = """
d      N
0.0000 13556
0.0097 10336
0.0195  8466
0.0293  6906
0.0390  6176
0.0487  5256
0.0586  4865
0.0689  4264
0.0791  3872
0.0896  3562
0.1002  3257
0.1112  3071
0.1225  2812
0.1410  2321
0.2810  1037
0.4150   496
0.5520   240
0.6931   159
0.8301   117
0.9616   119
1.0954   101
"""

# Načteme naměřená data z řetězce `data_str` do DataFrame, jehož sloupce jsou pojmenovány
#  - 'd' pro tloušťku absorbátoru,
#  - 'N' pro počet událostí.
# Přidáme sloupec 'N_unc' do DataFrame, který obsahuje neurčitost N.

from io import StringIO

# Vytvoříme pandas DataFrame z výše uvedeného řetězce.
df = pd.read_table(StringIO(data_str), sep = "\s+")

# Definujeme absorpční funkci.
def absorption_curve(d, N_1, mu_E1, N_2, mu_E2, N_B):
    return N_1 * np.exp(-mu_E1 * d) + N_2 * np.exp(-mu_E2 * d) + N_B

# Pro daná pole tloušťky d a počtu událostí N odhadneme hodnoty parametrů absorption_curve.
def estimate_parameters(d, N):
    nom, cov = curve_fit(absorption_curve, d, N, sigma = N ** 0.5, absolute_sigma = True)
    return nom

# Vypočítáme nominální odhady parametrů.
nom_estimates = estimate_parameters(df['d'], df['N'])

pseudoexperiments = []

# Vygenerujeme 100 pseudoexperimentů generováním četností N pro každou tloušťku d
# podle Poissonova rozdělení se střední hodnotou danou df['N'].
n_toys = 100
n_failed_fits = 0
for i in range(n_toys):
    if i % 10 == 0:
        print(f'Generuji pseudoexperiment {i} / {n_toys}')
    N_pseudo = np.random.poisson(df['N'])
    try:
        pseudo = estimate_parameters(df['d'], N_pseudo)
        # Prohodíme pořadí parametrů, aby odpovídalo pořadí v nom_estimates.
        if abs(pseudo[0] - nom_estimates[0]) > abs(pseudo[0] - nom_estimates[2]):
            pseudo = [pseudo[2], pseudo[3], pseudo[0], pseudo[1], pseudo[4]]
        pseudoexperiments.append(pseudo)
    except:
        print(f'Nepodařilo se fitovat pseudoexperiment {i}')
        n_failed_fits += 1
        
# pseudoexperiments je seznam 100 polí, každé pole obsahuje odhadované parametry
# pro jeden pseudoexperiment. Lze ho převést na numpy pole tvaru (100, 5).
pseudoexperiments = np.array(pseudoexperiments)


In [ ]:

# Vypočítáme kovarianční matici odhadovaných parametrů přes pseudoexperimenty.
cov_matrix = np.cov(pseudoexperiments, rowvar = False)

# Odhadneme kovarianční matici pomocí funkce curve_fit a porovnáme ji s kovarianční maticí
# odhadnutou pomocí pseudoexperimentů.
nom, cov = curve_fit(absorption_curve, df['d'], df['N'], sigma = df['N'] ** 0.5, absolute_sigma = True)

# Porovnání prvek po prvku obou kovarianční matic a jejich relativní rozdíly.
param_names = ['N_1', 'mu_E1', 'N_2', 'mu_E2', 'N_B']
idx = pd.MultiIndex.from_product([param_names, param_names], names=['row', 'col'])

comparison_df = pd.DataFrame(
    {
        'cov_matrix (toys)': cov_matrix.ravel(),
        'cov (curve_fit)': cov.ravel(),
    },
    index=idx
)

# Vypočítáme relativní rozdíl mezi prvky obou kovarianční matic
# a přidáme ho jako třetí sloupec do DataFrame.
den = comparison_df['cov (curve_fit)'].to_numpy()
num = (comparison_df['cov_matrix (toys)'] - comparison_df['cov (curve_fit)']).to_numpy()
comparison_df['rel. diff.'] = np.where(np.isclose(den, 0), np.nan, num / den)

# Zobrazíme tabulku.
display(
    comparison_df.style
    .format({
        'cov_matrix (toys)': '{:.3e}',
        'cov (curve_fit)': '{:.3e}',
        'rel. diff.': '{:+.2%}'
    })
    .set_caption('Porovnání kovariančních matic: pseudoexperimenty vs. curve_fit')
)


In [ ]:
# Dodatek: implementace vzorce pro odhad kovarianční matice ručně bez použití np.cov.
mean_params = np.mean(pseudoexperiments, axis = 0)
cov_matrix_manual = np.zeros((5, 5))
for i in range(n_toys - n_failed_fits):
    diff = pseudoexperiments[i] - mean_params
    cov_matrix_manual += np.outer(diff, diff)
cov_matrix_manual /= (n_toys - n_failed_fits - 1)